<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/_banners/opim5509_banner.svg" width="100%" alt="OPIM 5509 banner"/>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509-notebooks/blob/main/Module4/RNN_Samples_and_the_Hidden_State.ipynb)

# RNN Samples and the Hidden State
--------------------------------------------------
**Dr. Dave Wanik - University of Connecticut**

Two questions to settle **before** we count parameters or stack layers:

1. **What does a sample look like?** The lags notebook built a flat table. A recurrent layer wants a **3-D tensor**: `(samples, look-back, features)`. We'll build it with `split_sequence` and `split_sequences`, using a look-back of 5.
2. **What does the recurrent layer do with one sample?** It reads the 5 steps oldest to newest, updating its hidden state (the **red dots**) at every step. After the last step, the final red dots go to the dense layer. Or, with `return_sequences=True`, the whole sequence of red dots becomes the input to **another** recurrent layer.

Everything below is small enough to check by hand, and we will: every red dot is recomputed in NumPy and compared with Keras.

🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 1b — Making samples for an RNN: split_sequence and split_sequences
- The lags notebook gave us a flat table - fine for a dense net. An RNN needs (samples, look-back, features). This video is only about building that tensor.
- Play the GIF: a 12-value series, look-back 5, the green window slides one step at a time, the gold box is the target. 12 - 5 = 7 samples.
- Run split_sequence on the same 12 numbers and read the table: identical to the GIF. Then the reshape: (7, 5) -> (7, 5, 1). That trailing 1 is 'one feature' - forget it and Keras complains.
- Now three stocks (the three green dots): AAPL, MSFT, TSLA daily % changes, made-up numbers, plus a target column LAST: Netflix's price the next day (made-up prices). split_sequences -> (9, 5, 3). Look at one sample: 5 days by 3 stocks.
- Say the rule: samples = rows - look-back (+1 when the target sits in the same row, as it does here). Features = columns you feed in. Look-back = how many days each sample remembers.
-->


In [1]:
import numpy as np
import pandas as pd
import keras
from keras import Input
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
keras.utils.set_random_seed(5509)   # reproducibility: same weights, same numbers, every run

## Part 1 · Making samples

![making samples with a look-back of 5](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module4/img/m4_split_sequence_lookback5.gif)

### The window-making function: `split_sequence` (one column)

- **`n_steps`** is the **look-back**: each input X is `n_steps` values in a row, oldest first.
- The target y is the **next** value after the window.
- Every window slides forward by one step, so a series of length $N$ gives $N - n\_steps$ samples.

In [2]:
def split_sequence(sequence, n_steps):
    X, y = [], []
    for i in range(len(sequence) - n_steps):
        X.append(sequence[i:i + n_steps])      # the look-back window (oldest -> newest)
        y.append(sequence[i + n_steps])        # the value right after it
    return np.array(X), np.array(y)

series = [12, 14, 13, 17, 18, 16, 19, 21, 20, 22, 24, 23]    # same 12 numbers as the GIF
X, y = split_sequence(series, n_steps=5)

pd.DataFrame(X, columns=["t-4", "t-3", "t-2", "t-1", "t"], index=[f"sample {k+1}" for k in range(len(X))]).assign(**{"y (next)": y})

,t-4,t-3,t-2,t-1,t,y (next)
sample 1,12,14,13,17,18,16
sample 2,14,13,17,18,16,19
sample 3,13,17,18,16,19,21
sample 4,17,18,16,19,21,20
sample 5,18,16,19,21,20,22
sample 6,16,19,21,20,22,24
sample 7,19,21,20,22,24,23


In [3]:
print("X shape:", X.shape, "  <- (samples, look-back)")
X = X.reshape(X.shape[0], X.shape[1], 1)       # add the 'features' axis: one column = one feature
print("X shape:", X.shape, "<- (samples, look-back, features): what an RNN wants")
print("y shape:", y.shape)

X shape: (7, 5)   <- (samples, look-back)
X shape: (7, 5, 1) <- (samples, look-back, features): what an RNN wants
y shape: (7,)


### More than one column: three stocks

Now the three green dots: daily % changes for **AAPL, MSFT and TSLA** (made-up numbers, so the arithmetic stays readable). The target goes in the **last column**: **Netflix's price the next day** (NFLX, also made up). Because the target already looks one day ahead, the window can end on the same row as its target.

### The window-making function: `split_sequences` (many columns)

- **`n_steps`** is the **look-back**: X is the last `n_steps` rows of **every column except the target**.
- y is the **target column at the final row** of the window.
- Result: X is `(samples, n_steps, n_features)`, exactly what `input_shape=(n_steps, n_features)` expects.

In [4]:
stocks = pd.DataFrame({
    "AAPL": [0.5, -0.3, 1.2, 0.8, -0.4, 0.6, -1.1, 0.9, 0.3, -0.2, 1.4, -0.6, 0.7, 0.2],
    "MSFT": [0.2,  0.1, 0.9, 0.4, -0.2, 0.3, -0.8, 0.5, 0.1,  0.0, 1.0, -0.3, 0.4, 0.1],
    "TSLA": [1.5, -2.0, 3.1, 0.7, -1.1, 2.2, -3.0, 1.8, 0.9, -1.4, 2.6, -1.9, 1.2, 0.5]},
    index=[f"day {d}" for d in range(1, 15)])
nflx = [480.1, 478.9, 486.2, 489.0, 486.5, 491.3, 484.0, 489.8, 491.2, 489.5, 497.3, 493.1, 496.4, 497.6]
stocks["NFLX_tomorrow"] = pd.Series(nflx, index=stocks.index).shift(-1)   # target LAST: Netflix's price the next day
stocks = stocks.iloc[:-1]                                                   # day 14 has no 'tomorrow'
stocks

,AAPL,MSFT,TSLA,NFLX_tomorrow
day 1,0.5,0.2,1.5,478.9
day 2,-0.3,0.1,-2.0,486.2
day 3,1.2,0.9,3.1,489.0
day 4,0.8,0.4,0.7,486.5
day 5,-0.4,-0.2,-1.1,491.3
day 6,0.6,0.3,2.2,484.0
day 7,-1.1,-0.8,-3.0,489.8
day 8,0.9,0.5,1.8,491.2
day 9,0.3,0.1,0.9,489.5
day 10,-0.2,0.0,-1.4,497.3


In [5]:
def split_sequences(sequences, n_steps):
    X, y = [], []
    for i in range(len(sequences) - n_steps + 1):
        end = i + n_steps
        X.append(sequences[i:end, :-1])        # n_steps rows of every column except the target
        y.append(sequences[end - 1, -1])       # the target at the window's final row
    return np.array(X), np.array(y)

X3, y3 = split_sequences(stocks.values, n_steps=5)
print("X shape:", X3.shape, "<- (samples, look-back, features) = 13 rows - 5 + 1 = 9 samples, 5 days, 3 stocks")
print("y shape:", y3.shape)

X shape: (9, 5, 3) <- (samples, look-back, features) = 13 rows - 5 + 1 = 9 samples, 5 days, 3 stocks
y shape: (9,)


In [6]:
# one sample, the way the RNN will read it: 5 days (oldest first) x 3 stocks
sample = X3[0]
pd.DataFrame(sample, columns=["AAPL", "MSFT", "TSLA"], index=["t-4", "t-3", "t-2", "t-1", "t"]).assign(**{"NFLX tomorrow (target)": ["", "", "", "", y3[0]]})

,AAPL,MSFT,TSLA,NFLX tomorrow (target)
t-4,0.5,0.2,1.5,
t-3,-0.3,0.1,-2.0,
t-2,1.2,0.9,3.1,
t-1,0.8,0.4,0.7,
t,-0.4,-0.2,-1.1,491.3


🔴
<!-- 🎙 DAVE TALKING POINTS — M4 · 1c — What the red dots do: one step at a time, then Dense (or another RNN)
- One sample, 5 days x 3 stocks. The SimpleRNN has 2 hidden units: the two red dots. Play the GIF: they start at [0, 0], and at each day they're recomputed from TODAY's 3 green dots and the PREVIOUS red dots.
- The formula on screen: h_t = tanh(x_t W + h_{t-1} U + b). Same W, U, b at every step - one small network reused 5 times. That's why the parameter count ignores the look-back.
- Now the proof: pull W, U, b out of a Keras SimpleRNN, run the loop in NumPy, print the red dots after every day, and compare with Keras - identical.
- Default return_sequences=False: only the LAST pair goes on, shape (None, 2), into Dense(1) with a LINEAR activation: a number, Netflix's price. (In practice, scale the target first so that number lands in a sensible range.)
- Stacking: return_sequences=True keeps all 5 pairs, shape (None, 5, 2) - a new sequence, 2 features per step - and a second SimpleRNN reads it the same way. Its i is 2 (the first layer's red dots), so it has 2(2+2)+2 = 10 params, not 12. Total 12 + 10 + 3 = 25 - read it off summary().
-->


## Part 2 · What the red dots do

![a SimpleRNN(2) reading 5 days, then Dense](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module4/img/m4_simplernn_lookback5_to_dense.gif)

*The GIF uses a small made-up set of weights so the numbers are easy to read. The code below does exactly the same thing with the weights Keras starts with.*

At each step $t$, the two red dots $h_t$ are recomputed from **today's inputs** $x_t$ (the three green dots) and **the previous red dots** $h_{t-1}$:

$$
h_t \;=\; \tanh\big(\,x_t\,W \;+\; h_{t-1}\,U \;+\; b\,\big), \qquad h_0 = [\,0,\;0\,]
$$

| piece | shape here | what it is |
| :-- | :-- | :-- |
| $x_t$ | 3 | today's three stock returns (green dots) |
| $W$ | 3 × 2 | input weights: green dots → red dots |
| $U$ | 2 × 2 | recurrent weights: previous red dots → new red dots |
| $b$ | 2 | one bias per red dot |
| $h_t$ | 2 | the red dots after step $t$ |

The **same** $W$, $U$ and $b$ are used at every step. That's the "recurrent" part, and it's why a look-back of 5 or 5,000 needs the same number of weights.

In [7]:
# a SimpleRNN(2) that returns the red dots at EVERY step, so we can compare step by step
rnn_all = SimpleRNN(2, return_sequences=True)
keras_steps = rnn_all(sample[None, ...].astype("float32")).numpy()[0]      # (5, 2)
W, U, b = rnn_all.get_weights()
print("W (input -> red dots):", W.shape, "| U (red dots -> red dots):", U.shape, "| b:", b.shape)

# the same thing by hand, one day at a time
h = np.zeros(2)
rows = []
for t, x_t in enumerate(sample):
    h = np.tanh(x_t @ W + h @ U + b)
    rows.append({"day": ["t-4", "t-3", "t-2", "t-1", "t"][t], "AAPL": x_t[0], "MSFT": x_t[1], "TSLA": x_t[2],
                 "red dot 1": h[0], "red dot 2": h[1]})
by_hand = pd.DataFrame(rows).set_index("day")
print("by hand == Keras at every step:", np.allclose(by_hand[["red dot 1", "red dot 2"]].values, keras_steps, atol=1e-5))
by_hand.round(3)

W (input -> red dots): (3, 2) | U (red dots -> red dots): (2, 2) | b: (2,)
by hand == Keras at every step: True


,AAPL,MSFT,TSLA,red dot 1,red dot 2
day,,,,,
t-4,0.5,0.2,1.5,0.947,-0.925
t-3,-0.3,0.1,-2.0,-0.841,0.991
t-2,1.2,0.9,3.1,0.997,-1.000
t-1,0.8,0.4,0.7,0.969,-0.319
t,-0.4,-0.2,-1.1,-0.405,0.927


In [8]:
# the default layer (return_sequences=False) returns ONLY the final pair - and it's the same pair
rnn_last = SimpleRNN(2)
rnn_last.build((None, 5, 3)); rnn_last.set_weights([W, U, b])              # same weights as above
final = rnn_last(sample[None, ...].astype("float32")).numpy()[0]
print("final red dots from Keras:", final.round(3), "| last row of the table:", by_hand.iloc[-1, -2:].values.round(3))

# ...and those two numbers are all the dense layer ever sees
model = Sequential([Input((5, 3)), rnn_last, Dense(1)])            # Dense(1): linear activation, predicts a number
print("NFLX prediction from an untrained model:", model.predict(sample[None, ...], verbose=0).round(3)[0, 0],
      "(meaningless until it's trained - and you'd scale the ~490 prices first)")
model.summary()

final red dots from Keras:

 [-0.405  0.927] | last row of the table: [-0.405  0.927]


NFLX prediction from an untrained model: 0.834 (meaningless until it's trained - and you'd scale the ~490 prices first)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 2)              │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15 (60.00 B)

 Trainable params: 15 (60.00 B)

 Non-trainable params: 0 (0.00 B)

## Part 3 · Stacking: `return_sequences=True`

![stacking two SimpleRNN layers](https://raw.githubusercontent.com/drdave-teaching/OPIM5509-notebooks/main/Module4/img/m4_stacked_return_sequences.gif)

Set `return_sequences=True` and the first layer hands back **all 5 pairs** of red dots, shaped `(5, 2)`. That's a brand-new sequence: 5 steps, 2 features per step. A second recurrent layer reads it exactly the way the first one read the stock returns, one step at a time. Only the **last** recurrent layer drops `return_sequences`, so its final pair can go to `Dense`.

| layer | reads | returns | shape |
| :-- | :-- | :-- | :-- |
| `SimpleRNN(2, return_sequences=True)` | 5 steps × 3 stocks | a red-dot pair at **every** step | `(None, 5, 2)` |
| `SimpleRNN(2)` | 5 steps × 2 red dots | only its **final** pair | `(None, 2)` |
| `Dense(1)`, linear | 2 numbers | Netflix's price | `(None, 1)` |

In [9]:
stacked = Sequential([Input((5, 3)),
                      SimpleRNN(2, return_sequences=True),     # keeps every step's red dots
                      SimpleRNN(2),                            # reads that sequence, returns only the last pair
                      Dense(1)])                                # linear output: Netflix's price
stacked.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 5, 2)           │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25 (100.00 B)

 Trainable params: 25 (100.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# trace layer 2 by hand: its inputs are layer 1's red dots, one step at a time
W1, U1, b1 = stacked.layers[0].get_weights()
W2, U2, b2 = stacked.layers[1].get_weights()

h1, seq1 = np.zeros(2), []
for x_t in sample:                         # layer 1 reads the stocks
    h1 = np.tanh(x_t @ W1 + h1 @ U1 + b1); seq1.append(h1)
seq1 = np.array(seq1)                      # (5, 2): the sequence that return_sequences=True hands over

h2 = np.zeros(2)
for s_t in seq1:                           # layer 2 reads layer 1's red dots
    h2 = np.tanh(s_t @ W2 + h2 @ U2 + b2)

layer1_keras = stacked.layers[0](sample[None, ...].astype("float32")).numpy()[0]
layer2_keras = stacked.layers[1](layer1_keras[None, ...]).numpy()[0]
print("layer 1 sequence by hand == Keras:", np.allclose(seq1, layer1_keras, atol=1e-5), "| shape", seq1.shape)
print("layer 2 final pair by hand == Keras:", np.allclose(h2, layer2_keras, atol=1e-5), "|", h2.round(3))

layer 1 sequence by hand == Keras: True | shape (5, 2)
layer 2 final pair by hand == Keras: True | [-0.719 -0.115]


### ✏️ The parameter count, written out

Using the formula from *RNNs by Hand*, $g\,[\,h(h+i) + h\,]$ with $g = 1$ for a SimpleRNN:

$$
\begin{aligned}
\textbf{SimpleRNN}(2)\text{ on 3 stocks} & = 1\,[\,2(2+3) + 2\,] = \mathbf{12} \\
\textbf{SimpleRNN}(2)\text{ on 2 red dots} & = 1\,[\,2(2+2) + 2\,] = \mathbf{10} \\
\textbf{Dense}(1) & = 2 \cdot 1 + 1 = \mathbf{3} \\
\textbf{Total} & = 12 + 10 + 3 = \mathbf{25}
\end{aligned}
$$

The second layer's $i$ is **2**, the first layer's red dots, not the 3 stocks. That's the rule for every stack: **each layer's inputs are the previous layer's units.**

## What to remember

- An RNN sample is a small **table**: look-back rows × feature columns, oldest first. Many samples make the 3-D tensor `(samples, look-back, features)`.
- The recurrent layer reads that table **one row at a time**, recomputing its red dots from today's row and the previous red dots, with the **same weights** at every step.
- `return_sequences=False` (the default) passes on only the **final** red dots, ready for `Dense`.
- `return_sequences=True` passes on **every** step's red dots, a new sequence ready for **another** recurrent layer.

**Next:** *RNNs by Hand* counts the parameters for SimpleRNN, LSTM and GRU layers of any size.